- A00298173
- Data Visualisation 4 Project
- Submitted on 15/12/2024

**Part 1: Data Collection through Web Scraping**

- In this first part, we extracted the pokemon data from the pokedex. To do this, we first found the information we wanted.
- Implemented a 4 second delay to avoid overloading the server, then began scraping the pokedex table!
- We used requests to first fetch the webpage and BeautifulSoup to parse the HTML into the JSON format.
- We then saved the scraped data into that JSON file.

In [13]:
import requests
from bs4 import BeautifulSoup
import time
import json

In [14]:
# URL of the main Pokédex page
base_url = "https://pokemondb.net"
pokedex_url = f"{base_url}/pokedex/all"

# Fetch the main Pokédex page
response = requests.get(pokedex_url)
response.raise_for_status()  # Check for request success
soup = BeautifulSoup(response.text, "html.parser")

# Locate the table containing Pokémon
table = soup.find("table", {"id": "pokedex"})
rows = table.find_all("tr")[1:]  # Skip header row

# Extract Pokémon URLs
pokemon_links = []
for row in rows:
    link = row.find("a")["href"]
    pokemon_links.append(f"{base_url}{link}")

print(f"Found {len(pokemon_links)} Pokémon URLs.")


Found 1215 Pokémon URLs.


In [15]:
# Placeholder for Pokémon data
pokemon_data = {}

# Function to extract text from a row
def extract_stat(soup, label):
    stat_row = soup.find("th", text=label)
    return stat_row.find_next("td").text.strip() if stat_row else None

# Loop through Pokémon URLs
for i, link in enumerate(pokemon_links, start=1):
    # Respect scraping etiquette
    time.sleep(4)

    # Fetch the Pokémon's detailed page
    response = requests.get(link)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    # Extract Pokémon details
    name = soup.find("h1").text.strip().lower()
    national_no = extract_stat(soup, "National №")
    types = [t.text.strip() for t in soup.find_all("a", {"class": "type-icon"})]
    species = extract_stat(soup, "Species")
    height = extract_stat(soup, "Height")
    weight = extract_stat(soup, "Weight")
    abilities = [a.text.strip() for a in soup.find("th", text="Abilities").find_next("td").find_all("a")]
    ev_yield = extract_stat(soup, "EV yield")
    catch_rate = extract_stat(soup, "Catch rate")
    base_friendship = extract_stat(soup, "Base Friendship")
    base_exp = extract_stat(soup, "Base Exp")
    growth_rate = extract_stat(soup, "Growth Rate")
    egg_groups = extract_stat(soup, "Egg Groups")
    gender = extract_stat(soup, "Gender")
    egg_cycles = extract_stat(soup, "Egg cycles")

    # Evolution path
    evo_path_section = soup.find("h2", string="Evolution chain")
    evo_path = (
        [e.text.strip() for e in evo_path_section.find_next("ul").find_all("a")]
        if evo_path_section else []
    )

    # Moves
    moves_table = soup.find("table", {"class": "data-table"})
    moves = (
        [m.text.strip() for m in moves_table.find_all("a")] if moves_table else []
    )

    # Pixel images
    images = [
        img["src"] for img in soup.find_all("img") if "sprites" in img["src"]
    ]

    # Add data to dictionary
    pokemon_data[name] = {
        "name": name.capitalize(),
        "national_no": national_no,
        "types": types,
        "species": species,
        "height": height,
        "weight": weight,
        "abilities": abilities,
        "ev_yield": ev_yield,
        "catch_rate": catch_rate,
        "base_friendship": base_friendship,
        "base_exp": base_exp,
        "growth_rate": growth_rate,
        "egg_groups": egg_groups,
        "gender": gender,
        "egg_cycles": egg_cycles,
        "evo_path": evo_path,
        "moves": moves,
        "pixel_image_urls": images,
    }

    print(f"Scraped {name.capitalize()} ({i}/{len(pokemon_links)})")

# Save the data to a JSON file
with open("pokemon_data.json", "w") as f:
    json.dump(pokemon_data, f, indent=4)


<ipython-input-15-e1d07901fe70>:6: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  stat_row = soup.find("th", text=label)
<ipython-input-15-e1d07901fe70>:26: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  abilities = [a.text.strip() for a in soup.find("th", text="Abilities").find_next("td").find_all("a")]


Scraped Bulbasaur (1/1215)
Scraped Ivysaur (2/1215)
Scraped Venusaur (3/1215)
Scraped Venusaur (4/1215)
Scraped Charmander (5/1215)
Scraped Charmeleon (6/1215)
Scraped Charizard (7/1215)
Scraped Charizard (8/1215)
Scraped Charizard (9/1215)
Scraped Squirtle (10/1215)
Scraped Wartortle (11/1215)
Scraped Blastoise (12/1215)
Scraped Blastoise (13/1215)
Scraped Caterpie (14/1215)
Scraped Metapod (15/1215)
Scraped Butterfree (16/1215)
Scraped Weedle (17/1215)
Scraped Kakuna (18/1215)
Scraped Beedrill (19/1215)
Scraped Beedrill (20/1215)
Scraped Pidgey (21/1215)
Scraped Pidgeotto (22/1215)
Scraped Pidgeot (23/1215)
Scraped Pidgeot (24/1215)
Scraped Rattata (25/1215)
Scraped Rattata (26/1215)
Scraped Raticate (27/1215)
Scraped Raticate (28/1215)
Scraped Spearow (29/1215)
Scraped Fearow (30/1215)
Scraped Ekans (31/1215)
Scraped Arbok (32/1215)
Scraped Pikachu (33/1215)
Scraped Pikachu (34/1215)
Scraped Raichu (35/1215)
Scraped Raichu (36/1215)
Scraped Sandshrew (37/1215)
Scraped Sandshrew (38/

**Part 2: Data Loading and Preprocessing**
- To clean and structure the JSON dat for analysis by loading it into a Pandas DataFrame and performing necessary preprocessing steps to facilitate analysis.
- First, I loaded the JSON data and converted it into a Pandas DataFrame using pd.DataFrame.from_dict.
- I split types into two columns, primary and secondary. And abilities into primary and secondary.
- Then, I extracted male and females % from the gender field and made two new columns for the two of them.
Finally, I removed any redundant columns after splitting and saved it into a pickle file.


In [16]:
import pandas as pd
import json

# Load JSON file
with open('pokemon_data.json', 'r') as f:
    data = json.load(f)

# Convert to Pandas DataFrame
df = pd.DataFrame.from_dict(data, orient='index')


In [17]:
# Convert `national_no` and `base_exp` to integers
df['national_no'] = pd.to_numeric(df['national_no'], errors='coerce')
df['base_exp'] = pd.to_numeric(df['base_exp'], errors='coerce')

# Extract numerical values from `catch_rate`
df['catch_rate'] = df['catch_rate'].str.extract('(\d+)').astype(float)

# Convert height and weight to numeric (meters and kilograms)
df['height_m'] = df['height'].str.extract('(\d+\.\d+)').astype(float)
df['weight_kg'] = df['weight'].str.extract('(\d+\.\d+)').astype(float)


In [18]:
df['primary_type'] = df['types'].apply(lambda x: x[0] if x else None)
df['secondary_type'] = df['types'].apply(lambda x: x[1] if len(x) > 1 else 'None')
df.drop(columns=['types'], inplace=True)  # Drop the original types column


In [19]:
df['male_percentage'] = df['gender'].str.extract('(\d+\.\d+|\d+)(?=% male)').astype(float)
df['female_percentage'] = df['gender'].str.extract('(\d+\.\d+|\d+)(?=% female)').astype(float)
df.drop(columns=['gender'], inplace=True)  # Drop the original gender column


In [20]:
def assign_generation(national_no):
    if 1 <= national_no <= 151:
        return 1
    elif 152 <= national_no <= 251:
        return 2
    elif 252 <= national_no <= 386:
        return 3
    elif 387 <= national_no <= 493:
        return 4
    elif 494 <= national_no <= 649:
        return 5
    elif 650 <= national_no <= 721:
        return 6
    elif 722 <= national_no <= 809:
        return 7
    elif 810 <= national_no <= 905:
        return 8
    else:
        return None

df['generation'] = df['national_no'].apply(assign_generation)


In [21]:
df['primary_ability'] = df['abilities'].apply(lambda x: x[0] if x else None)
df['secondary_ability'] = df['abilities'].apply(lambda x: x[1] if len(x) > 1 else 'None')
df.drop(columns=['abilities'], inplace=True)  # Drop the original abilities column


In [22]:
df.to_pickle('cleaned_pokemon_data.pkl')


In [23]:
# Check datatypes and null values
print(df.info())

# View unique values in categorical columns
print(df['primary_type'].unique())
print(df['secondary_type'].unique())


<class 'pandas.core.frame.DataFrame'>
Index: 1025 entries, bulbasaur to pecharunt
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   name               1025 non-null   object 
 1   national_no        1025 non-null   int64  
 2   species            1025 non-null   object 
 3   height             1025 non-null   object 
 4   weight             1025 non-null   object 
 5   ev_yield           1025 non-null   object 
 6   catch_rate         1025 non-null   float64
 7   base_friendship    0 non-null      object 
 8   base_exp           0 non-null      float64
 9   growth_rate        1025 non-null   object 
 10  egg_groups         1025 non-null   object 
 11  egg_cycles         1025 non-null   object 
 12  evo_path           1025 non-null   object 
 13  moves              1025 non-null   object 
 14  pixel_image_urls   1025 non-null   object 
 15  height_m           1025 non-null   float64
 16  weight_kg       

**Part 3: Analysis of Pokémon Distribution by Primary Type**
- In this step, the aim was to sort the pokemon counts for each primary type using the value_counts method.#
- To do that, I mapped each pokemon to a colour value for data visualation!
- I used ploty to make myself a bar chart, sorted each type by colour and finally just added some titles and labels to help make the chart more clear and concise.
- Doing this, we can now analyse which pokemon tpye is the most and least common.

In [24]:
import plotly.graph_objects as go

# Count by primary type
type_counts = df['primary_type'].value_counts()

# Define type colors
type_colors = {
    'Grass': '#78C850', 'Fire': '#F08030', 'Water': '#6890F0',
    'Bug': '#A8B820', 'Normal': '#A8A878', 'Poison': '#A040A0',
    'Electric': '#F8D030', 'Ground': '#E0C068', 'Fairy': '#EE99AC',
    'Fighting': '#C03028', 'Flying': '#A890F0', 'Psychic': '#F85888',
    'Rock': '#B8A038', 'Ghost': '#705898', 'Ice': '#98D8D8',
    'Dragon': '#7038F8', 'Dark': '#705848', 'Steel': '#B8B8D0'
}

# Create bar chart
fig = go.Figure(data=[
    go.Bar(
        x=type_counts.index,
        y=type_counts.values,
        marker_color=[type_colors.get(t, '#A8A878') for t in type_counts.index]
    )
])

# Layout
fig.update_layout(
    title="Pokémon Count by Primary Type",
    xaxis_title="Primary Type",
    yaxis_title="Count"
)

fig.show()


**Part 4: Comparative Analysis of Pokémon Distribution by Primary Type and Generation**
- This part was to explore how pokemon types are distributed across different generations.
- Here, I organized the data by grouping the generation and primary_type. By doing so I also counted how many of each type within each generation using groupby and size.
- As same as before, I would use plotly to make grouped bar charts for each type in the generation. Also using a similar colour scheme.
- Though this time, I would use a stacked bar chart. Mostly just for visual purposes as it makes the comparison across generations much more clear.
- By analsying these graphs, we can see dragons are much more prevalent in the newer generations!

In [25]:
# Count Pokémon by primary type and generation
type_gen_counts = df.groupby(['generation', 'primary_type']).size().unstack(fill_value=0)

# Create grouped bar chart
fig = go.Figure()

for ptype in type_gen_counts.columns:
    fig.add_trace(
        go.Bar(
            x=type_gen_counts.index,
            y=type_gen_counts[ptype],
            name=ptype,
            marker_color=type_colors.get(ptype, '#A8A878')
        )
    )

# Layout
fig.update_layout(
    title="Pokémon Distribution by Type and Generation",
    xaxis_title="Generation",
    yaxis_title="Count",
    barmode="stack"
)

fig.show()


**Height and Weight**
- Here, I made a scatter plot to visualise height and weight relationships.
- I used height_m and weight_kg columns for plotting.
- Added hover tooltips to display all the names.
- We can see in the chart most of the pokemons are clustered together, leaving some serious outliers on the graph for extremely heavy and tall pokemons.

In [26]:
# Scatter plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df['height_m'],
    y=df['weight_kg'],
    mode='markers',
    marker=dict(
        size=8,
        color=df['primary_type'].map(type_colors),
        opacity=0.7
    ),
    text=df['name']  # Tooltip
))

fig.update_layout(
    title="Pokémon Height vs Weight",
    xaxis_title="Height (m)",
    yaxis_title="Weight (kg)"
)

fig.show()


**Stats**
- This step involved visualising the stats of the pokemon. Their total stats across the generations.
- First, I summed up the stats (I had some problems in scraping however.) into a new column.
- Put together a box plot using plotly and used colours for reference once again.


In [28]:
import numpy as np

# Calculate total stats
df['total_stats'] = df[['hp_lv1', 'atk_lv1', 'def_lv1', 'sat_lv1', 'sde_lv1', 'spd_lv1']].sum(axis=1)

# Box plot
fig = go.Figure()

for gen in sorted(df['generation'].unique()):
    gen_data = df[df['generation'] == gen]['total_stats']
    fig.add_trace(go.Box(
        y=gen_data,
        name=f"Gen {gen}",
        marker_color='#FFA07A'
    ))

fig.update_layout(
    title="Distribution of Total Stats by Generation",
    yaxis_title="Total Stats"
)

fig.show()


Index(['name', 'national_no', 'species', 'height', 'weight', 'ev_yield',
       'catch_rate', 'base_friendship', 'base_exp', 'growth_rate',
       'egg_groups', 'egg_cycles', 'evo_path', 'moves', 'pixel_image_urls',
       'height_m', 'weight_kg', 'primary_type', 'secondary_type',
       'male_percentage', 'female_percentage', 'generation', 'primary_ability',
       'secondary_ability'],
      dtype='object')


KeyError: "None of [Index(['hp_lv1', 'atk_lv1', 'def_lv1', 'sat_lv1', 'sde_lv1', 'spd_lv1'], dtype='object')] are in the [columns]"